# Train MemGraph Brain: small local Librarian + Archivist LoRA for Home Mind

This trains **Qwen3-0.6B** (or **Qwen3-1.7B**) with LoRA on your Home Mind memory graph flywheel dataset (`[RETRIEVE]` router + `[INGEST]` archivist patches, secret-scrubbed 70/30 corrections + golden replay mix).

1. In Home Mind → **System › Models › Train small models**, click **Train in Colab** next to **MemGraph Brain** (this downloads `memgraph_training.json` and opens this notebook).
2. Select **Runtime › Change runtime type › T4** (free) and click **Runtime › Run all**.
3. Upload `memgraph_training.json` when prompted. At the end, `memgraph-lora.gguf` downloads automatically with the exact `Modelfile` and `config.yaml` snippet.


In [ ]:
%pip install -q "torch" "transformers>=4.51" "datasets" "accelerate" "trl>=0.19" "peft>=0.14"


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4, L4 or A100"
print(torch.cuda.get_device_name(0))


## 1. Upload `memgraph_training.json` (or `.jsonl`) from Home Mind

In [ ]:
import json
from google.colab import files
up = files.upload()  # pick memgraph_training.json downloaded from Home Mind
fname, raw = next(iter(up.items()))
text = raw.decode('utf-8')
if fname.endswith('.jsonl'):
    records = [json.loads(line) for line in text.splitlines() if line.strip()]
    bundle = {'records': records}
else:
    bundle = json.loads(text)
    records = bundle.get('records', [])
LIBRARIAN_SYSTEM = bundle.get('librarian_system', 'You are the memory graph Librarian. Reply with JSON only.')
ARCHIVIST_SYSTEM = bundle.get('archivist_system', 'You are the memory graph Archivist. Reply with JSON only.')
print(f'Loaded {len(records)} records (fresh={sum(not r.get("golden") for r in records)}, replay={sum(bool(r.get("golden")) for r in records)})')


## 2. Build chat dataset & score baseline before training

In [ ]:
import random, re
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = bundle.get('base_model') or 'Qwen/Qwen3-0.6B'  # swap to 'Qwen/Qwen3-1.7B' for higher accuracy
EPOCHS = 3

def to_row(rec):
    sys_p = LIBRARIAN_SYSTEM if rec.get('mode') == '[RETRIEVE]' else ARCHIVIST_SYSTEM
    return {
        'messages': [
            {'role': 'system', 'content': sys_p},
            {'role': 'user', 'content': f"{rec.get('mode', '[INGEST]')} {rec.get('input', '')}"},
            {'role': 'assistant', 'content': json.dumps(rec.get('output', {}), ensure_ascii=False)},
        ],
        'golden': bool(rec.get('golden')),
        'expected': rec.get('output', {}),
    }

rows = [to_row(r) for r in records]
random.Random(7).shuffle(rows)
cut = max(2, min(len(rows) // 5, 40))
val_rows, train_rows = rows[:cut], rows[cut:]
golden_rows = [r for r in rows if r['golden']][:20]

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=dtype, device_map='auto')

def score_split(m, split):
    if not split:
        return {'accuracy': 1.0, 'hallucination_rate': 0.0}
    ok = hall = 0
    for r in split:
        ids = tok.apply_chat_template(r['messages'][:2], add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to(m.device)
        with torch.no_grad():
            out = m.generate(ids, max_new_tokens=200, do_sample=False)
        txt = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        m_json = re.search(r'\{.*\}', txt, re.S)
        try:
            parsed = json.loads(m_json.group(0)) if m_json else None
        except Exception:
            parsed = None
        exp = r['expected']
        if not isinstance(parsed, dict):
            hall += 1
            continue
        if 'patches' in exp:
            ep, gp = exp.get('patches') or [], parsed.get('patches') or []
            if not ep and not gp:
                ok += 1
            elif not ep and gp:
                hall += 1
            elif ep and gp:
                ok += int(str(ep[0].get('relation', '')).upper() == str(gp[0].get('relation', '')).upper())
        else:
            ok += int(bool(parsed.keys() & exp.keys()))
    return {'accuracy': round(ok / len(split), 4), 'hallucination_rate': round(hall / len(split), 4)}

b_val, b_gold = score_split(model, val_rows), score_split(model, golden_rows)
baseline = {'accuracy': b_val['accuracy'], 'golden_score': b_gold['accuracy'], 'hallucination_rate': b_val['hallucination_rate']}
print('Baseline:', baseline)


## 3. Fine-tune LoRA & run Home Mind's regression gate

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ds = Dataset.from_list([{'messages': r['messages']} for r in train_rows])
args = SFTConfig(
    output_dir='memgraph-lora', num_train_epochs=EPOCHS, per_device_train_batch_size=4,
    gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05,
    logging_steps=10, save_strategy='no', max_length=1024,
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(), report_to=[],
)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM')
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tok, peft_config=lora)
trainer.train()

merged = trainer.model.merge_and_unload()
c_val, c_gold = score_split(merged, val_rows), score_split(merged, golden_rows)
candidate = {'accuracy': c_val['accuracy'], 'golden_score': c_gold['accuracy'], 'hallucination_rate': c_val['hallucination_rate']}
reasons = []
if candidate['accuracy'] - baseline['accuracy'] < 0.02 - 1e-9:
    reasons.append(f"gain {candidate['accuracy'] - baseline['accuracy']:+.3f} < +0.020")
if candidate['golden_score'] < baseline['golden_score'] - 1e-9:
    reasons.append(f"golden regressed ({candidate['golden_score']:.3f} < {baseline['golden_score']:.3f})")
if candidate['hallucination_rate'] > 0.05 + 1e-9:
    reasons.append(f"hallucination {candidate['hallucination_rate']:.3f} > 0.050")
print('Candidate:', candidate)
print('Regression gate:', 'PROMOTE' if not reasons else 'HOLD -> ' + '; '.join(reasons))


## 4. Export `memgraph-lora.gguf` for Ollama

In [ ]:
merged.save_pretrained('memgraph-lora-merged')
tok.save_pretrained('memgraph-lora-merged')
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py memgraph-lora-merged --outtype q8_0 --outfile memgraph-lora.gguf
!ls -lh memgraph-lora.gguf
files.download('memgraph-lora.gguf')


## 5. Run it in Ollama

Put `memgraph-lora.gguf` in a folder with a `Modelfile`:

```
FROM ./memgraph-lora.gguf
PARAMETER temperature 0.0
PARAMETER num_predict 256
```

Run `ollama create memgraph-lora -f Modelfile`, then set in Home Mind's `config.yaml`:

```yaml
models:
  - name: memgraph_small
    type: ollama
    url: http://YOUR_OLLAMA_IP:11434
    model: memgraph-lora
librarian_model: memgraph_small
archivist_model: memgraph_small
```
